## Redes Neurais Artificiais 2025.2

- **Disciplina**: Redes Neurais Artificiais 2025.2
- **Professora**: Elloá B. Guedes (ebgcosta@uea.edu.br)  
- **Github**: http://github.com/elloa  

# Importação das Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import os
import time
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, classification_report

import joblib

# Importação da base de dados:

In [3]:
path = "dataset_tratado.csv"

data = pd.read_csv(path)

# Elaboração da Grade de busca e Hiperparâmetros

## 1. Algumas observações
- O dataset é pequeno (2221 linhas), o treinamento dos modelos é rápido, portanto é possível treinar várias combinações.
- Temos 33 dados de entrade e 1 de saída


## 2. Regras de neurônios (33 entradas, 1 saída)

### 2.1) Pirâmide geométrica
Fórmula:
$$n_{oculto}=α\sqrt{n_{entrada}\times n_{saida}}$$
$$0.5 < α < 2$$
Aplicação:
$$n_{oculto}=α\sqrt{33\times1}=α\sqrt{33}\approx5.74α$$
$$5.74 * 0.5 < 5.74*α < 5.74 * 2$$
$$2.87 < n_{oculto} < 11.48$$
$$3 \le n_{oculto} \le 11$$
**Resultado:** Entre 3 e 11 neurônios

---

### 2.2) Regra para Neurônios com Uma Camada Oculta
Fórmula:
$$0.5*n_{entrada}\le n_{oculto}\le3* n_{entrada}$$
Aplicação:
$$0.5*33\le n_{oculto}\le3* 33$$
$$16.5\le n_{oculto}\le99$$

**Resultado:** Entre 17 e 99 neurônios

Vamos levar em consideração 3 a 11 neurônios.

## 3. Criando a grid de busca

In [5]:
# Definindo 32 arquiteturas de 1 e 2 camadas, de acordo com a analise anterior
arquiteturas_32 = [
    # 1 camada
    (3,), (4,), (5,), (6,), (7,), (8,), (9,), (10,), (11,),
    # 2 camadas
    (1,2), (2,1),
    (1,3), (2,2),
    (1,4), (2,3), (3,2),
    (1,5), (2,4), (3,3),
    (1,6), (2,5), (3,4),
    (1,7), (2,6), (3,5),
    (1,9), (3,6), (7,2),
    (1,10), (4,6), (6,4), (10, 1)
]
print(f"Tem que ser 32 -> Len = {len(arquiteturas_32)}")

Tem que ser 32 -> Len = 32


In [6]:
# Eduardo vai rodar essa
grade_adam_18_432 = {
    'hidden_layer_sizes': arquiteturas_32,                          # 32
    'activation': ['relu', 'logistic', 'tanh'],                     # 3
    'solver': ['adam'],                                             # 1
    'learning_rate_init': [0.0001, 0.001, 0.01],                    # 3
    'batch_size': [20, 32, 100, 128],                               # 4
    'n_iter_no_change': [10, 20, 30, 50],                           # 4
    'max_iter': [50, 100, 200, 300]                                 # 4
}
# Alexandro vai rodar essa
grade_sgd_20_736 = {
    'hidden_layer_sizes': arquiteturas_32,                          # 32
    'activation': ['relu', 'logistic', 'tanh'],                     # 3
    'solver': ['sgd'],                                              # 1
    'learning_rate_init': [0.0001, 0.001, 0.01],                    # 3
    'learning_rate': ['adaptive'],                                  # 1
    'momentum': [0.8, 0.9],                                         # 2
    'batch_size': [32, 128, 256],                                   # 3
    'n_iter_no_change': [10, 20, 50],                               # 3
    'max_iter': [50, 100, 200, 300]                                 # 4
}
grade_teste_12 = {
    'solver': ['adam'],
    'batch_size': [50, 100, 200],
    'learning_rate_init': [0.001, 0.01],
    'n_iter_no_change': [20,30],
    'max_iter': [500],
    'activation': ['relu']
}

# Treinamento dos modelos

In [7]:
# Preparando os dados
coluna_alvo = 'NObeyesdad'
X = data.drop(columns=[coluna_alvo])
Y = data[coluna_alvo]

# Separando dados para treino e teste
X_treino, X_teste, Y_treino, Y_teste = train_test_split(
    X, Y, test_size=0.15, random_state=42
)

In [ ]:
# Rodando o Grid Search
modelo = MLPClassifier()
grid = GridSearchCV(
    estimator=modelo,
    param_grid=grade_sgd_20_736,
    scoring='f1_macro', # parametro de avaliacao
    cv=5, # Numero de folds na validacao cruzada
    verbose=1,
    n_jobs=-1
)

nome_de_quem_rodou = "Alexandro" #Eduardo, Alexandro
if nome_de_quem_rodou == "":
    raise Exception("Falta atualizar com o nome de quem rodou\nVerifique também se alterou o 'param_grid' do GridSearchCV")

#Treino
tempo_inicio_total = time.time()
grid.fit(X_treino, Y_treino)
tempo_total = time.time() - tempo_inicio_total

In [ ]:
# Definindo caminho base para salvar os resultados
caminho_base = os.path.join("Resultados", nome_de_quem_rodou)
os.makedirs(caminho_base, exist_ok=True)

# Salvando o resultado do treino
resultado = grid.cv_results_

df = pd.DataFrame(resultado)
df.to_csv(os.path.join(caminho_base,"cv_results.csv"), index=False)

indices_ordenados = np.argsort(resultado['mean_test_score'])[::-1]
tops = {
    "melhores":{
        "idx": indices_ordenados[:10],
        "nome_pasta" : "10-Melhores",
        "info_json": "10-Melhores-Info"
    },
    "piores":{
        "idx": indices_ordenados[-10:][::-1],
        "nome_pasta" : "10-Piores",
        "info_json": "10-Piores-Info"
    }
}

print("\n=== TOP 10 MODELOS ===")
for top in tops:
    indices = tops[top]["idx"]
    nome_pasta = tops[top]["nome_pasta"]
    nome_info_json = tops[top]["info_json"]
    os.makedirs(os.path.join(caminho_base, nome_pasta), exist_ok=True)
    info_json = []
    for rank, idx in enumerate(indices, start=1):
        params = resultado['params'][idx]
        score = resultado['mean_test_score'][idx]
        std = resultado['std_test_score'][idx]

        print(f"\n#{rank}")
        print("Hiperparâmetros:", params)
        print(f"Score médio: {score:.4f} ± {std:.4f}")

        # Treinar o modelo com esses hiperparâmetros
        modelo = grid.estimator.set_params(**params)
        modelo.fit(X_treino, Y_treino)

        # Nome do arquivo
        nome_arquivo = os.path.join(caminho_base, nome_pasta, f"modelo_top{rank}.joblib")
        joblib.dump(modelo, nome_arquivo)

        # Adicionar ao JSON
        info_json.append({
            "posicao": rank,
            "hiperparametros": params,
            "score_medio": float(score),
            "score_std": float(std),
            "arquivo_modelo": nome_arquivo
        })

    # Salvar JSON
    caminho_json = os.path.join(caminho_base, nome_info_json + ".json")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(info_json, f, indent=4, ensure_ascii=False)
    print("\nJSON salvo em:", caminho_json)
print(f"Tempo total de treinamento: {tempo_total:.2f}s")